All raw data used for the project are available for viewing at https://github.com/vk4444/DATA301_Project/tree/main

In [2]:
# @title Installations

!pip install -U pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 44.3 MB/s eta 0:00:00


In [30]:
# @title Imports

import dask.bag as db
import dask.dataframe as ddf
import dask.array as da
import pandas as pd
import numpy as np
import pypdfium2 as pdfium
import re

# Load datasets into the notebook

In [4]:
# @title Book File Names
book_file_names = ['babicka_bozena_nemcova',
'windows8_redakce_businessIT_a_partneri',
'cesky_rozhlas-historie_eva_jesutova_a_kolektiv',
'flvek_05_alois_jirasek',
'flvek_4_alois_jirasek',
'flvek_03_alois_jirasek',
'flvek_02_alois_jirasek',
'flvek_01_alois_jirasek',
'jihoslovanske_jazyky_pavel_krejci',
'nase_nynejsi_krise_tomas_garrigue_masaryk',
'lucerna_alois_jirasek',
'matka_karel_capek',
'basne_josef_vaclav_sladek',
'dalimilova_kronika_dalimil',
'obycejny_zivot_karel_capek',
'povetron_karel_capek',
'hordubal_karel_capek',
'noc_na_karlstejne_jaroslav_vrchlicky',
'pisne_kosmicke_jan_neruda',
'domaci_kucharka_magdalena_dobromila_rettigova',
'hovory_s_tg_masarykem_karel_capek',
'dramaticke_zlomky_karel_hynek_macha',
'povidani_o_pejskovi_a_kocicce_josef_capek',
'obrazy_z_dejin_naroda_ceskeho_iii_vladislav_vancura',
'obrazy_z_dejin_naroda_ceskeho_ii_vladislav_vancura',
'obrazy_z_dejin_naroda_ceskeho_i_vladislav_vancura',
'rur_karel_capek',
'bylo_nas_pet_karel_polacek',
'mistr_kampanus_zikmund_winter',
'konec_starych_casu_vladislav_vancura',
'ballady_a_romance_jan_neruda',
'vec_makropulos_karel_capek',
'krakatit_karel_capek',
'filosofska_historie_alois_jirasek',
'stare_povesti_ceske_alois_jirasek',
'devatero_pohadek_karel_capek',
'kosmuw_letopis_cesky_kosmas',
'tezka_hodina_jiri_wolker',
'host_do_domu_jiri_wolker',
'nova_evropa_tomas_garrigue_masaryk',
'sedm_let_v_jizni_africe_iv_emil_holub',
'sedm_let_v_jizni_africe_iii_emil_holub',
'sedm_let_v_jizni_africe_druha_cesta_emil_holub',
'sedm_let_v_jizni_africe_prvni_cesta_emil_holub',
'bila_nemoc_karel_capek',
'rozmarne_leto_vladislav_vancura',
'maj_karel_hynek_macha',
'kytice_karel_jaromir_erben',
'osudy_dobreho_vojaka_svejka_jaroslav_hasek_volumes3and4',
'osudy_dobreho_vojaka_svejka_jaroslav_hasek_volumes1and2',
'broucci_jan_karafiat',
]

This is the dictionnary of abbreviations that will be expanded. This is done so that the text corresponds to how it would be pronounced and to correctly separate the sentences later. The dictionnary was adapted from https://cja.ujc.cas.cz/e-cja/zkratky which lists some of the most common abbreviations in the Czech Language. The dictionnary is not exhaustive and some of the abbreviations might have different interpretations in certain contexts.

In [5]:
# @title Dictionnary of Abbreviations

#adapted from https://cja.ujc.cas.cz/e-cja/zkratky
abbr_dict = {
    "adj.": "adjektivum",
    "adv.": "adverbium",
    "aj.": "a jiné",
    "akuz.": "akuzativ",
    "apod.": "a podobně",
    "atd.": "a tak dále",
    "býv.": "bývalý",
    "č.": "č",
    "čes.": "český",
    "dat.": "dativ",
    "dolož.": "doloženo",
    "doudl.": "doudlebský",
    "dř.": "dříve",
    "f.": "femininum",
    "gen.": "genitiv",
    "imp.": "imperativ",
    "ind.": "indikativ",
    "inf.": "infinitiv",
    "inform.": "informátor",
    "instr.": "instrumentál",
    "jč.": "jihočeský",
    "již.": "jižní",
    "jjv.": "jihojihovýchodní",
    "jjz.": "jihojihozápadní",
    "jv.": "jihovýchod",
    "jz.": "jihozápad",
    "jzč.": "jihozápadočeský",
    "km": "kilometr",
    "kol.": "kolektiv",
    "kond.": "kondicionál",
    "lid.": "lidový",
    "lok.": "lokál",
    "m.": "maskulinum",
    "m n. m.": "metry nad mořem",
    "min.": "minulý",
    "n. l.": "našeho letopočtu",
    "např.": "například",
    "nar.": "narozen",
    "nář.": "nářečí",
    "nepřízv.": "nepřízvučný",
    "neživ.": "neživotný",
    "nom.": "ominativ",
    "obl.": "oblast",
    "obyv.": "obyvatel",
    "okr.": "okres",
    "os.": "osoba",
    "pl.": "plurál",
    "plt.": "plurale tantum",
    "poč.": "počátek",
    "popř.": "popřípadě",
    "préz.": "prézens",
    "protet.": "protetický",
    "předp.": "předpona",
    "přech.": "přechodník",
    "příč.": "příčestí",
    "příp.": "přípona",
    "přísl.": "příslovce",
    "přít.": "přítomný",
    "přivl.": "přivlastňovací",
    "př. n. l.": "před naším letopočtem",
    "pův.": "původní",
    "r.": "rok",
    "s.": "strana",
    "samohl.": "samohláska",
    "sev.": "severní",
    "sg.": "singulár",
    "slez.": "slezský",
    "souhl.": "souhláska",
    "ssv.": "severoseverovýchodní",
    "ssz.": "severoseverozápadní",
    "stol.": "století",
    "střč.": "středočeský",
    "střm.": "středomoravský",
    "subst.": "substantivum",
    "sv.": "svatý",
    "svč.": "severovýchodočeský",
    "sz.": "severozápad",
    "tj.": "to je",
    "trp.": "trpný",
    "tř.": "třída",
    "tzn.": "to znamená",
    "tzv.": "takzvaný",
    "ukaz.": "ukazovací",
    "vjv.": "východojihovýchodní",
    "vm.": "východomoravský",
    "vok.": "vokativ",
    "vsv.": "východoseverovýchodní",
    "vých.": "východní",
    "zájm.": "zájmeno",
    "záp.": "západní",
    "zč.": "západočeský",
    "zjz.": "západojihozápadní",
    "zsz.": "západoseverozápadní",
    "zvl.": "zvláště",
    "zvrat.": "zvratný",
    "živ.": "životný"
}


In [6]:
# @title Utility Functions

# loads books and converts from pdf to txt
def book_loader(filename: str) -> tuple[str, str]:
  document = pdfium.PdfDocument('https://raw.githubusercontent.com/vk4444/DATA301_Project/main/books/' + filename + '.pdf')

  version_marker_found = 0
  book_started = 0
  text = ''
  for page in document:
    # extract text from the page
    textpage = page.get_textpage()
    extractedtext = textpage.get_text_bounded()

    # the following two conditions ensure that the material attached to the book that is not a part of the original text is skipped (for example the cover page, info about publication etc.)
    if version_marker_found == 0 and 'verze' in extractedtext.lower():
      version_marker_found = 1
      print('version marker found')

    # after the version marker is found (indicating the last page of added material), the next page is checked for containing the contents (obsah) of the book which can also be skipped.
    elif version_marker_found and book_started == 0:
      if 'obsah' not in extractedtext.lower():
        book_started = 1

    if book_started:
      text += extractedtext

  return filename, text

# generates urls for all the wiki files
def generate_wiki_urls(last_chr: str = 'N', last_number: int = 55) -> list[str]:
  urls = []
  current_chr = 'A'
  current_num_str = '00'

  # generates urls for all the files
  while ord(current_chr) <= ord(last_chr):
    n_of_files = 100

    if ord(current_chr) == ord(last_chr):
      n_of_files = last_number + 1

    for i in range(n_of_files):
      if i < 10:
        current_num_str = '0' + str(i)
      else:
        current_num_str = str(i)

      urls.append('https://raw.githubusercontent.com/vk4444/DATA301_Project/main/wiki/A' + current_chr + '/' + 'wiki_' + current_num_str)

    current_chr = chr(ord(current_chr) + 1)

  return urls

In [7]:
# @title Loading the Data
# load the books
books = db.from_sequence(book_file_names).map(book_loader)

# load the Czech Wikipedia
wiki = (
    db.read_text(generate_wiki_urls())
    .filter(lambda x: x[:4] != '<doc' and '__NOEDITSECTION__' not in x and '</doc>' not in x)
    .map(lambda x: ('wiki', x))
)

survey_anonymized = ddf.read_csv('https://raw.githubusercontent.com/vk4444/DATA301_Project/main/survey/anonymized_data.csv', na_values=['<NA>'])

all_text = db.concat([wiki, books])

In [8]:
# verify all files have loaded correctly
assert books.npartitions == 51
assert wiki.npartitions == 1356

In [9]:
# look at the first few items of text
print(all_text.take(15))

(('wiki', 'Hlavní strana\n'), ('wiki', '\n'), ('wiki', 'internetové encyklopedii, kterou může .&lt;br&gt;Česká Wikipedie má nyní .\n'), ('wiki', '&lt;br&gt;&lt;br&gt;\n'), ('wiki', ' • • \n'), ('wiki', ' • \n'), ('wiki', 'Ostatní projekty\n'), ('wiki', 'Další informace…\n'), ('wiki', ' • \n'), ('wiki', '. v minulosti\n'), ('wiki', '\n'), ('wiki', 'Astronomie\n'), ('wiki', '\n'), ('wiki', 'Astronomie, řecky αστρονομία z άστρον (astron) hvězda a νόμος (nomos) zákon, česky též hvězdářství, je věda, která se zabývá jevy za hranicemi zemské atmosféry. Zvláště tedy výzkumem vesmírných těles, jejich soustav, různých dějů ve vesmíru i vesmírem jako celkem.\n'), ('wiki', 'Historie astronomie.\n'))


In [10]:
# look at the first few rows of the survey
survey_anonymized.head()

,Unnamed: 0,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,...,A19_1,A20_1,A21_1,A22_1,A23_1,A24_1,A25_1,A26_1,A27_1,A28_1
0,0,Start Date,End Date,Response Type,anonymized,Progress,Duration (in seconds),Finished,Recorded Date,Response ID,...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...,Focus on the consonant in the audio. How soft ...
1,1,"{""ImportId"":""startDate"",""timeZone"":""America/De...","{""ImportId"":""endDate"",""timeZone"":""America/Denv...","{""ImportId"":""status""}",anonymized,"{""ImportId"":""progress""}","{""ImportId"":""duration""}","{""ImportId"":""finished""}","{""ImportId"":""recordedDate"",""timeZone"":""America...","{""ImportId"":""_recordId""}",...,"{""ImportId"":""QID23_1""}","{""ImportId"":""QID24_1""}","{""ImportId"":""QID25_1""}","{""ImportId"":""QID26_1""}","{""ImportId"":""QID27_1""}","{""ImportId"":""QID28_1""}","{""ImportId"":""QID29_1""}","{""ImportId"":""QID30_1""}","{""ImportId"":""QID31_1""}","{""ImportId"":""QID32_1""}"
2,2,2026-05-10 17:11:47,2026-05-10 17:13:19,Survey Preview,anonymized,100,91,True,2026-05-10 17:13:19,R_4JE55jXT7iFK35L,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,3,2026-05-10 18:20:56,2026-05-10 18:26:17,IP Address,anonymized,100,320,True,2026-05-10 18:26:18,R_9ZV8QTRGmaUyEur,...,87,87,80,75,68,77,<NA>,19,61,77
4,4,2026-05-10 18:37:15,2026-05-10 18:43:13,IP Address,anonymized,100,358,True,2026-05-10 18:43:14,R_9M4FuHomfPqmXye,...,61,63,22,38,71,70,65,86,95,95


# Data Pre-processing

In [11]:
# @title Utility Functions

# a function that removes every expression in a list from the given string
def clean_text(text: str, expr: list[str]) -> str:
  for e in expr:
    text = text.replace(e, '')
    print(e)

  return text

# splits a body of text into sentences
def to_sentences(item: tuple) -> list[tuple[str, str]]:
  name = item[0]
  text = item[1]
  sentences = []

  for sentence in re.split('\\.|\\!|\\?', text):
    sentences.append((name, sentence))

  return sentences

# converts dictionnary keys to regular expressions that look for the expression followed by non upper case letter character
def abbr_to_regex(expr: str) -> str:
  return expr.replace('.', '\\.') + '(?=\\s*[^A-Z])'

# expands abbreviations to their full form
def expand_abbr(text: str) -> str:
  for abbr in sorted(abbr_dict, key=len, reverse=True):
    regex = abbr_to_regex(abbr)
    text = re.sub(regex, abbr_dict[abbr], text)

  return text

# removes stopping signs that are used outside of the context of the end of sentence as determined by the following non-space character.
def remove_non_ending_stops(text) -> str:
    regex = '(\\.|\\!|\\:|\\;|\\?)(?=\\s*[^A-Z])'
    text = re.sub(regex, '', text)

    return text

# converts "no"/"yes" string values to boolean
def yes_no_to_binary(text: str) -> int:
  if text == 'Yes':
    return 1
  else:
    return 0

def df_to_num(df):
  for col in df:
    df[col] = pd.to_numeric(df[col], errors='coerce')

  return df


In [12]:
# @title Pre-Processing Pipelines

# clean the text pipeline
expr_to_clean = ['\n', '\r', ',', '-', '–', '—', ';', '“', '0', '1', '2', '3', '4', '5', '6','7','8','9','\x02', '(', ')', '„', '•', '&ltbr', '&gt']
pre_processed_text = (
    all_text.map(lambda x: (x[0], clean_text(x[1], expr_to_clean))) # removes specified expressions
    .map(lambda x: (x[0], expand_abbr(x[1]))) # expands abbreviations
    .map(lambda x: (x[0], remove_non_ending_stops(x[1]))) # removes sentence ending characters outside the context of a sentence end
    .map(lambda x: to_sentences(x)).flatten() # splits items into sentences
    .map(lambda x: (x[0], x[1].strip())) # strips trailing spaces
    .filter(lambda x: x[1] != '') # excludes empty strings
    .filter(lambda x: len(x[1].split()) >= 3) # excludes sentences that have less than 3 words
    .filter(lambda x: x[1][0].isupper()) # excludes sentences that start with a lowercase character and therefore are likely incomplete
)

# survey data preprocessing pipeline
pre_processed_survey = (survey_anonymized.loc[:, 'D1':'A28_1'] # responses to the questions
                        [survey_anonymized['Finished'] == 'True'] # only those participants who have finished the survey
                        [survey_anonymized['DistributionChannel'] == 'anonymous'] # only production distribution of the survey (excludes previews)
                        )
pre_processed_survey['D1'] = pre_processed_survey['D1'].replace({'Yes': '1', 'No': '0'})
pre_processed_survey = pre_processed_survey.map_partitions(df_to_num)



In [13]:
pre_processed_survey.head()

,D1,A1_1,A2_1,A3_1,A4_1,A5_1,A6_1,A7_1,A8_1,A9_1,...,A19_1,A20_1,A21_1,A22_1,A23_1,A24_1,A25_1,A26_1,A27_1,A28_1
3,0,53,14,48,0,45,20,38,50,14,...,87,87,80,75,68,77,<NA>,19,61,77
4,0,87,69,63,33,52,43,40,83,37,...,61,63,22,38,71,70,65,86,95,95
5,1,11,5,3,72,71,94,5,38,46,...,57,39,100,83,24,61,82,68,16,7
6,0,96,37,19,20,18,15,15,58,11,...,91,54,69,73,75,87,21,52,84,43
7,1,60,70,60,30,80,40,60,70,50,...,80,80,40,80,30,<NA>,30,20,60,<NA>


In [14]:
pre_processed_survey.dtypes

,0
D1,Float64
A1_1,Float64
A2_1,Float64
A3_1,Float64
A4_1,Float64
A5_1,Float64
A6_1,Float64
A7_1,Float64
A8_1,Float64
A9_1,Float64


In [15]:
print(pre_processed_text.take(5))

(('wiki', 'Astronomie řecky αστρονομία z άστρον astron hvězda a νόμος nomos zákon česky též hvězdářství je věda která se zabývá jevy za hranicemi zemské atmosféry Zvláště tedy výzkumem vesmírných těles jejich soustav různých dějů ve vesmíru i vesmírem jako celkem'), ('wiki', 'Astronomie se podobně jako další vědy začala rozvíjet ve starověku Na území Babylonie však nebylo k popisu používáno již vynalezené geometrie grafy První se z astronomie rozvíjela astrometrie zabývající se měřením poloh hvězd a planet na obloze Tato oblast astronomie měla velký význam pro navigaci Podstatnou částí astrometrie je sférická astronomie sloužící k popisu poloh objektů na nebeské sféře zavádí souřadnice a popisuje významné křivky a body na nebeské sféře Pojmy ze sférické astronomie se také používají při měření času'), ('wiki', 'Další oblastí astronomie která se rozvinula byla nebeská mechanika Zabývá se pohybem těles v gravitačním poli například planet ve sluneční soustavě Základem nebeské mechaniky jso

# Survey Analysis and processing

In [29]:
pre_processed_survey.compute().describe()

,D1,A1_1,A2_1,A3_1,A4_1,A5_1,A6_1,A7_1,A8_1,A9_1,...,A19_1,A20_1,A21_1,A22_1,A23_1,A24_1,A25_1,A26_1,A27_1,A28_1
count,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,...,17.0,17.0,17.0,17.0,16.0,16.0,16.0,17.0,16.0,15.0
mean,0.529412,58.823529,47.058824,36.705882,31.705882,60.941176,56.470588,26.882353,55.764706,38.529412,...,61.470588,55.235294,73.235294,65.941176,41.125,69.5625,41.4375,35.588235,58.625,59.4
std,0.514496,21.53844,22.859546,22.725989,24.692521,28.862758,29.879168,18.472555,25.498847,21.610523,...,24.832231,24.136408,21.294158,21.355534,24.824383,25.049202,28.774917,25.159637,22.138579,30.509483
min,0.0,11.0,5.0,3.0,0.0,14.0,15.0,5.0,6.0,11.0,...,13.0,11.0,22.0,23.0,8.0,5.0,3.0,4.0,16.0,7.0
25%,0.0,50.0,37.0,19.0,15.0,45.0,30.0,14.0,38.0,17.0,...,49.0,38.0,62.0,49.0,13.75,64.75,18.25,15.0,45.75,40.5
50%,1.0,61.0,52.0,33.0,30.0,60.0,50.0,19.0,52.0,40.0,...,64.0,50.0,71.0,73.0,46.5,73.0,43.0,25.0,60.5,68.0
75%,1.0,70.0,62.0,52.0,50.0,85.0,86.0,40.0,80.0,50.0,...,80.0,70.0,90.0,80.0,62.5,84.0,61.25,52.0,73.0,81.5
max,1.0,96.0,80.0,76.0,76.0,100.0,100.0,69.0,100.0,92.0,...,100.0,100.0,100.0,100.0,75.0,100.0,96.0,86.0,95.0,100.0


# Text analysis

In [60]:
# @title Utility Functions
# def softness():

# generates a matrix of means for each of the questions A1 - A28. Also gives the opportunity to use responses by Czech native speakers only.
def generate_matrix_of_means(survey: ddf, only_czech: bool = False):

  if only_czech:
    survey = survey[survey.D1 == 1]

  data = survey.loc[:, 'A1_1':'A28_1']
  means = data.mean()
  array = da.from_array(means.compute().to_list())

  return array.compute()


In [59]:
generate_matrix_of_means(pre_processed_survey)

array([58.82352941, 47.05882353, 36.70588235, 31.70588235, 60.94117647,
       56.47058824, 26.88235294, 55.76470588, 38.52941176, 48.25      ,
       30.76470588, 57.52941176, 45.58823529, 28.        , 37.75      ,
       50.23529412, 36.46666667, 33.        , 61.47058824, 55.23529412,
       73.23529412, 65.94117647, 41.125     , 69.5625    , 41.4375    ,
       35.58823529, 58.625     , 59.4       ])

# Separately used scripts

Originally, the Czech Wikipedia dump file consist of one big compressed file with Wikipedia's original formatting. The following script separates the dump file into multiple smaller chunks, each containing several articles, and extracts the text from the article. The script was run separately, because WikiExtractor needed to be run in an older environment.

In [ ]:
# @title WikiExtractor

# !python3.10 -m wikiextractor.WikiExtractor \ --output extracted \ cswiki-latest-pages-articles.xml.bz2

The raw survey dataset contained some personally identifiable information such as name, IP addresses and location. To maintain the original format of the dataframe for the purposes of this project, while keeping personally identifiable data private, the following script was used separately to anonymize the raw data downloaded from Qualtrics

In [ ]:
# @title Survey Data Anonymizer

# import pandas as pd

# data = pd.read_csv('raw_data.csv')
# data[['IPAddress', 'RecipientLastName', 'RecipientFirstName', 'RecipientEmail', 'LocationLatitude', 'LocationLongitude']] = 'anonymized'
# data.to_csv('anonymized_data.csv')